# ArSL Word Training — KArSL-502 Pre-extracted Keypoints (Fast Mode)
**No MediaPipe. No image processing. No pip install.**  
Loads pre-extracted `.npy` hand keypoints directly → trains BiLSTM.  
Dataset build: **~5 minutes** instead of hours.


In [ ]:
# CELL 1: IMPORTS
import os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import mixed_precision

warnings.filterwarnings('ignore')
print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print('Imports OK — No MediaPipe required!')



In [2]:
#karim part 

str el mashro3 accuracy = 0;
if(el mashro3 accuracy <= 50% );
el mashro3 accuracy = mashro3 accuracy + 50%; 





SyntaxError: invalid syntax (1048681011.py, line 3)

In [ ]:

# CELL 2: GPU SETUP
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    print(f'GPU: {gpus[0].name}')
else:
    print('No GPU — running on CPU (training will be slower)')
mixed_precision.set_global_policy('float32')


In [ ]:
# CELL 3: CONFIGURATION

IS_KAGGLE  = os.path.exists('/kaggle')
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else Path(r'M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True) if not IS_KAGGLE else None

# ── Hyper-parameters ──────────────────────────────────────────────
SEQUENCE_LENGTH = 48
BATCH_SIZE      = 64
EPOCHS          = 150
LEARNING_RATE   = 5e-4
LSTM_UNITS_1    = 256
LSTM_UNITS_2    = 128
LSTM_UNITS_3    = 64
DENSE_UNITS     = 256
DROPOUT_RATE    = 0.4
TEST_SIZE       = 0.4

# ── Hardcoded paths ───────────────────────────────────────────────
if IS_KAGGLE:
    KARSL_ROOT  = Path('/kaggle/input/blablabla/karsl-502')
    LABELS_FILE = '/kaggle/input/datasets/ahmed171102/karsl-502-labels/KARSL-502_Labels.txt'
else:
    KARSL_ROOT  = Path(r'M:\Term 10\Grad\SLR Main\Words\Datasets\KArSL_502')
    LABELS_FILE = str(Path(r'M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\KARSL-502_Labels.txt'))

# ── Verify ────────────────────────────────────────────────────────
if not KARSL_ROOT.exists():
    raise FileNotFoundError(f'Dataset not found: {KARSL_ROOT}')

print(f'KArSL root   : {KARSL_ROOT}  OK')
print(f'Labels file  : {LABELS_FILE}  {"OK" if os.path.exists(LABELS_FILE) else "NOT FOUND"}')
print(f'Output dir   : {OUTPUT_DIR}')
print(f'Sequence len : {SEQUENCE_LENGTH}')
print(f'Running on   : {"Kaggle" if IS_KAGGLE else "Local"}')


In [ ]:
# CELL 4: LOAD LABEL NAMES
id_to_english = {}
id_to_arabic  = {}

if LABELS_FILE and os.path.exists(str(LABELS_FILE)):
    with open(LABELS_FILE, 'r', encoding='utf-8', errors='replace') as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith('SignID'):
                continue
            parts = line.split('\t')
            if len(parts) >= 3:
                try:
                    sid = int(parts[0])
                    ar  = parts[1].strip()
                    en  = parts[2].strip()
                    id_to_english[sid] = en if en and en not in ('?','??','') else str(sid)
                    id_to_arabic[sid]  = ar if ar and ar not in ('?','??','') else en
                except:
                    continue
    print(f'Labels loaded: {len(id_to_english)} entries')
    print(f'Sample: {list(id_to_english.items())[:5]}')
else:
    print('Labels file not found — numeric IDs will be used as class names')


In [ ]:
# CELL 5: BUILD RECORDING MAP
# Structure: KARSL_ROOT/{signer}/{split}/{class_id}/lh_keypoints/
# e.g. /kaggle/input/blablabla/karsl-502/01/test/0289/lh_keypoints/

print('=' * 60)
print('BUILDING RECORDING MAP')
print('=' * 60)

class_recordings = {}
FEATURE_DIM = None

top_entries = sorted([e.name for e in os.scandir(str(KARSL_ROOT)) if e.is_dir()])
print(f'Signer folders: {top_entries}')

for signer in top_entries:
    signer_path = KARSL_ROOT / signer   # e.g. 01/  (no double folder)

    for split in ['train', 'test']:
        split_path = signer_path / split
        if not split_path.exists():
            continue

        for cls_entry in os.scandir(str(split_path)):
            if not cls_entry.is_dir() or not cls_entry.name.isdigit():
                continue

            class_id = int(cls_entry.name)
            lh_dir   = Path(cls_entry.path) / 'lh_keypoints'
            rh_dir   = Path(cls_entry.path) / 'rh_keypoints'

            if not lh_dir.exists() or not rh_dir.exists():
                continue

            lh_map = {Path(p.path).stem: p.path for p in os.scandir(str(lh_dir)) if p.name.endswith('.npy')}
            rh_map = {Path(p.path).stem: p.path for p in os.scandir(str(rh_dir)) if p.name.endswith('.npy')}
            common = set(lh_map) & set(rh_map)
            if not common:
                continue

            if class_id not in class_recordings:
                class_recordings[class_id] = []
            for stem in sorted(common):
                class_recordings[class_id].append((lh_map[stem], rh_map[stem]))
                if FEATURE_DIM is None:
                    try:
                        arr = np.load(lh_map[stem])
                        FEATURE_DIM = arr.shape[-1] if arr.ndim >= 2 else len(arr)
                    except:
                        pass

if not class_recordings:
    print('\nERROR: No lh_keypoints / rh_keypoints found!')
    raise FileNotFoundError('No .npy keypoint files found.')

class_ids    = sorted(class_recordings.keys())
NUM_FEATURES = (FEATURE_DIM or 21) * 2
for cid in class_ids:
    id_to_english.setdefault(cid, str(cid))
    id_to_arabic.setdefault(cid, str(cid))

total_recs = sum(len(v) for v in class_recordings.values())
named      = sum(1 for c in class_ids if id_to_english[c] != str(c))

print(f'\nClasses found  : {len(class_ids)} ({named} with named labels)')
print(f'Total rec pairs: {total_recs}')
print(f'Avg per class  : {total_recs / max(len(class_ids),1):.1f}')
print(f'Feature dim    : {FEATURE_DIM} per hand -> {NUM_FEATURES} total')
print(f'\nFirst 10 classes:')
for cid in class_ids[:10]:
    print(f'  {cid:4d}  {id_to_english[cid]:25s}  {len(class_recordings[cid])} recordings')


In [ ]:
# CELL 6: SANITY CHECK — verify one full sample loads correctly
print('=' * 60)
print('SANITY CHECK')
print('=' * 60)

_cid = class_ids[0]
_lh, _rh = class_recordings[_cid][0]
print(f'Class {_cid} ({id_to_english[_cid]})')
print(f'LH file : {os.path.basename(_lh)}')
print(f'RH file : {os.path.basename(_rh)}')

_lh_arr = np.load(_lh)
_rh_arr = np.load(_rh)
print(f'LH shape: {_lh_arr.shape}')
print(f'RH shape: {_rh_arr.shape}')

_combined = np.concatenate([_lh_arr, _rh_arr], axis=1) if _lh_arr.ndim > 1 else np.concatenate([_lh_arr, _rh_arr])
print(f'Combined: {_combined.shape}')
print(f'Sample values: {_combined.flatten()[:6]}')
print('SANITY CHECK PASSED!')


In [ ]:
# CELL 7: HELPER FUNCTIONS

def pad_or_sample(arr, target_len=SEQUENCE_LENGTH, target_feat=NUM_FEATURES):
    arr = arr.astype(np.float32)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    # Fix features
    if arr.shape[1] > target_feat:
        arr = arr[:, :target_feat]
    elif arr.shape[1] < target_feat:
        arr = np.concatenate([arr, np.zeros((arr.shape[0], target_feat - arr.shape[1]), dtype=np.float32)], axis=1)
    # Fix time
    if arr.shape[0] >= target_len:
        arr = arr[np.linspace(0, arr.shape[0]-1, target_len, dtype=int)]
    else:
        arr = np.concatenate([arr, np.zeros((target_len - arr.shape[0], target_feat), dtype=np.float32)], axis=0)
    return arr  # (SEQUENCE_LENGTH, NUM_FEATURES)


def load_sequence(lh_path, rh_path):
    try:
        lh = np.load(lh_path)
        rh = np.load(rh_path)
    except Exception as e:
        return None
    if lh.ndim == 1: lh = lh.reshape(1, -1)
    if rh.ndim == 1: rh = rh.reshape(1, -1)
    n = min(lh.shape[0], rh.shape[0])
    if n < 3:
        return None
    combined = np.concatenate([lh[:n], rh[:n]], axis=1)
    seq = pad_or_sample(combined)
    blank = np.sum(np.all(seq == 0, axis=1)) / len(seq)
    return None if blank > 0.8 else seq


print(f'Helpers ready | SEQUENCE_LENGTH={SEQUENCE_LENGTH} | NUM_FEATURES={NUM_FEATURES}')


In [ ]:
# CELL 8: BUILD DATASET FROM .NPY KEYPOINTS (or load cache)
print('=' * 60)
print('BUILDING DATASET')
print('=' * 60)

NPZ_PATH = OUTPUT_DIR / 'arsl_word_sequences_keypoints.npz'

if NPZ_PATH.exists():
    print(f'Cache found: {NPZ_PATH}')
    _d = np.load(NPZ_PATH)
    X, y = _d['X'], _d['y']
    print(f'X shape : {X.shape}')
    print(f'y shape : {y.shape}')
    print(f'Classes : {len(np.unique(y))}')
else:
    n_cls  = len(class_ids)
    n_recs = sum(len(v) for v in class_recordings.values())
    print(f'Classes  : {n_cls}')
    print(f'Rec pairs: {n_recs}\n')
    print(f'{"IDX":>5} {"ID":>5} {"Label":<22} {"Recs":>5} {"OK":>5} {"Skip":>5} {"Total":>7} {"Elapsed":>8} {"ETA":>8}')
    print('-' * 82)

    start  = time.time()
    X_list, y_list = [], []
    total_proc = skipped = 0

    for ci, class_id in enumerate(class_ids):
        pairs = class_recordings[class_id]
        label = id_to_english.get(class_id, str(class_id))
        ok = skip = 0

        for lh_path, rh_path in pairs:
            total_proc += 1
            seq = load_sequence(lh_path, rh_path)
            if seq is None:
                skipped += 1; skip += 1
            else:
                X_list.append(seq); y_list.append(class_id); ok += 1

        elapsed = time.time() - start
        rate    = (ci + 1) / elapsed if elapsed > 0 else 1e-9
        eta     = (n_cls - ci - 1) / rate
        print(f'{ci+1:5d} {class_id:5d} {label[:22]:<22} {len(pairs):5d} {ok:5d} {skip:5d} {len(X_list):7d} {elapsed/60:7.1f}m {eta/60:7.1f}m')

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)

    elapsed = time.time() - start
    print('-' * 82)
    print(f'\nDone in {elapsed:.1f}s ({elapsed/60:.2f} min)')
    print(f'X shape      : {X.shape}')
    print(f'Skipped      : {skipped} / {total_proc}')

    if len(X_list) == 0:
        raise RuntimeError('No samples extracted! Check dataset structure in Cell 5.')

    np.savez_compressed(NPZ_PATH, X=X, y=y)
    print(f'Saved: {NPZ_PATH}')


In [ ]:
# CELL 9: PREPROCESSING & SPLIT
print('=' * 60)
print('PREPROCESSING & SPLIT')
print('=' * 60)

_d  = np.load(NPZ_PATH)
X, y = _d['X'], _d['y']

# StandardScaler
orig  = X.shape
X_flat = X.reshape(-1, NUM_FEATURES)
scaler = StandardScaler()
X_flat = scaler.fit_transform(X_flat)
X = X_flat.reshape(orig).astype(np.float32)
np.savez_compressed(str(OUTPUT_DIR / 'arsl_scaler_stats.npz'),
                    mean=scaler.mean_.astype(np.float32),
                    scale=scaler.scale_.astype(np.float32))
print('Scaler saved')

# Encode labels
encoder   = LabelEncoder()
y_encoded = encoder.fit_transform(y)
num_classes = len(encoder.classes_)
y_onehot  = to_categorical(y_encoded, num_classes=num_classes)

# Save class map with real names
classes_df = pd.DataFrame({
    'model_class_index': range(num_classes),
    'label_name'   : [id_to_english.get(int(encoder.classes_[i]), str(encoder.classes_[i])) for i in range(num_classes)],
    'arabic_name'  : [id_to_arabic.get(int(encoder.classes_[i]),  str(encoder.classes_[i])) for i in range(num_classes)],
    'source_class_id': [int(c) for c in encoder.classes_]
})
classes_df.to_csv(str(OUTPUT_DIR / 'arsl_word_classes.csv'), index=False)
print(f'Class map saved ({num_classes} classes)')
print(classes_df.head(10).to_string())

# 60/20/20 split
try:
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y_onehot, test_size=TEST_SIZE, random_state=42, stratify=y_encoded)
    X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=np.argmax(y_tmp,1))
except ValueError:
    print('WARNING: Stratified split failed — using random split')
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y_onehot, test_size=TEST_SIZE, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

print(f'\nTrain : {X_tr.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'Classes: {num_classes}')


In [ ]:
# CELL 10: BUILD & TRAIN BiLSTM
print('=' * 60)
print('TRAINING BiLSTM')
print('=' * 60)

tf.keras.backend.clear_session()

model = Sequential([
    Bidirectional(LSTM(LSTM_UNITS_1, return_sequences=True),
                  input_shape=(SEQUENCE_LENGTH, NUM_FEATURES)),
    BatchNormalization(), Dropout(DROPOUT_RATE),
    Bidirectional(LSTM(LSTM_UNITS_2, return_sequences=True)),
    BatchNormalization(), Dropout(DROPOUT_RATE),
    LSTM(LSTM_UNITS_3, return_sequences=False),
    BatchNormalization(), Dropout(DROPOUT_RATE),
    Dense(DENSE_UNITS, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax', dtype='float32')
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')]
)
model.summary()

MODEL_BEST  = str(OUTPUT_DIR / 'arsl_word_lstm_best.h5')
MODEL_FINAL = str(OUTPUT_DIR / 'arsl_word_lstm_final.h5')

callbacks = [
    ModelCheckpoint(MODEL_BEST, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1, min_lr=1e-6),
]

_train_lbl = np.argmax(y_tr, axis=1)
_cw = dict(enumerate(compute_class_weight('balanced', classes=np.unique(_train_lbl), y=_train_lbl)))

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=callbacks, class_weight=_cw, verbose=1
)
model.save(MODEL_FINAL)
print(f'Saved best  : {MODEL_BEST}')
print(f'Saved final : {MODEL_FINAL}')


In [ ]:
# CELL 11: EVALUATION
print('=' * 60)
print('EVALUATION')
print('=' * 60)

best = tf.keras.models.load_model(MODEL_BEST)
proba  = best.predict(X_test, verbose=0)
y_pred = np.argmax(proba, axis=1)
y_true = np.argmax(y_test, axis=1)

top1 = (y_pred == y_true).mean()
top5 = sum(1 for i in range(len(y_true)) if y_true[i] in np.argsort(proba[i])[-5:]) / len(y_true)

print(f'Top-1 Accuracy : {top1*100:.2f}%')
print(f'Top-5 Accuracy : {top5*100:.2f}%')

word_labels = [id_to_english.get(int(encoder.classes_[i]), str(encoder.classes_[i])) for i in range(num_classes)]
print(classification_report(y_true, y_pred, target_names=word_labels, zero_division=0))

# Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.suptitle(f'Top-1: {top1*100:.1f}%  Top-5: {top5*100:.1f}%', fontsize=14)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=150)
plt.show()

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(cm, annot=False, cmap='Blues',
            xticklabels=word_labels, yticklabels=word_labels, ax=ax)
ax.set_title(f'Confusion Matrix — {num_classes} classes')
plt.xticks(rotation=90, fontsize=4); plt.yticks(fontsize=4)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'confusion_matrix.png'), dpi=150)
plt.show()
